In [ ]:
import asyncio
import random
from datetime import datetime
import redis.asyncio as redis
import nest_asyncio

nest_asyncio.apply()

num_test_streams = 3
pub_freq = 1
stream_max_len = 100

async def publish_test_data_for_stream(stream_index, redis_client):
    last_price = 100.0  # Starting price
    stream_key = f"test_{stream_index}"  # Using test_1, test_2, etc.

    while True:
        # Simulate large swings by adding more volatility
        change = random.uniform(-5, 5)  # Increased fluctuation range
        last_price = max(10, last_price + change)  # Keep price above zero

        # Force RSI boundary conditions sometimes
        if random.random() < 0.1:  
            last_price *= random.choice([0.85, 1.15])  # Big jumps 15% up or down

        # Create fake OHLC data
        data = {
            "symbol": "TEST",
            "timestamp": datetime.utcnow().isoformat(),
            "open": round(last_price - random.uniform(0.5, 2), 2),
            "high": round(last_price + random.uniform(0.5, 2), 2),
            "low": round(last_price - random.uniform(1, 3), 2),
            "close": round(last_price, 2),
            "volume": random.randint(100, 1000),
            "trade_count": random.randint(10, 50),
            "vwap": round(last_price + random.uniform(-1, 1), 2),
        }

        # Push data to Redis stream
        await redis_client.xadd(stream_key, data, maxlen=stream_max_len)
        print(f"Pushed to {stream_key}: {data}")

        await asyncio.sleep(pub_freq)  # Adjust frequency if needed

async def publish_test_data(num_streams=1):
    redis_client = redis.Redis(host='localhost', port=6379, decode_responses=True)

    # Create a list of tasks to run multiple streams concurrently
    tasks = []
    for stream_index in range(1, num_streams + 1):
        task = asyncio.create_task(publish_test_data_for_stream(stream_index, redis_client))
        tasks.append(task)

    # Run all the tasks concurrently
    await asyncio.gather(*tasks)


await publish_test_data(num_streams=num_test_streams)

C:\Users\Mano\AppData\Local\Temp\ipykernel_4208\200926913.py:29: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


Pushed to test_1: {'symbol': 'TEST', 'timestamp': '2025-03-18T12:31:11.747359', 'open': 100.29, 'high': 103.47, 'low': 99.6, 'close': 101.79, 'volume': 902, 'trade_count': 24, 'vwap': 101.39}
Pushed to test_2: {'symbol': 'TEST', 'timestamp': '2025-03-18T12:31:11.747359', 'open': 102.55, 'high': 105.26, 'low': 102.83, 'close': 104.46, 'volume': 412, 'trade_count': 22, 'vwap': 104.61}
Pushed to test_3: {'symbol': 'TEST', 'timestamp': '2025-03-18T12:31:11.749279', 'open': 94.89, 'high': 96.68, 'low': 94.4, 'close': 96.06, 'volume': 683, 'trade_count': 37, 'vwap': 95.07}
Pushed to test_2: {'symbol': 'TEST', 'timestamp': '2025-03-18T12:31:12.768099', 'open': 103.05, 'high': 104.96, 'low': 101.31, 'close': 104.13, 'volume': 974, 'trade_count': 46, 'vwap': 104.87}
Pushed to test_3: {'symbol': 'TEST', 'timestamp': '2025-03-18T12:31:12.768099', 'open': 96.71, 'high': 100.17, 'low': 97.37, 'close': 98.55, 'volume': 460, 'trade_count': 17, 'vwap': 98.45}
Pushed to test_1: {'symbol': 'TEST', 'time